### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="mice_protein_trisomy_discriminant",
    dataset_year="2015",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C50S3Z",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/342/mice+protein+expression.zip && unzip mice+protein+expression.zip && rm -rf mice+protein+expression.zip
mkdir -p local-data-warehouse/mice_protein_trisomy_discriminant && mv Data_Cortex_Nuclear.xls local-data-warehouse/mice_protein_trisomy_discriminant/
""",
    # References
    academic_reference_bibtex="""@article{higuera2015self,
  title={Self-organizing feature maps identify proteins critical to learning in a mouse model of down syndrome},
  author={Higuera, Clara and Gardiner, Katheleen J and Cios, Krzysztof J},
  journal={PloS one},
  volume={10},
  number={6},
  pages={e0129126},
  year={2015},
  publisher={Public Library of Science San Francisco, CA USA}
}
""",
    academic_reference_bibtex_key="higuera2015self",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We start with the dataset from UCI.

- We drop the three feature that make up the target to remove the leakage.
- We reduce the mouse ID to represent the ID of the mouse instead of the experiment for the mouse.
- We drop the duplicated feature pS6_N.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="class",
    group_on="MouseID",
    group_labels="per_group",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_excel(dataset_mold.path / "Data_Cortex_Nuclear.xls")
print("Loaded data shape:", df.shape)

# Fix the ID so it reduces to the same mouse
df["MouseID"] = df["MouseID"].str.split("_").str[0]
# Drop leaky features that are part of the target
df = df.drop(columns=["Genotype", "Treatment", "Behavior"])
as_type_cat = ["MouseID", "class"]
df[as_type_cat] = df[as_type_cat].astype("category")
# Drop duplicated column
df = df.drop(columns=["pS6_N"])
df = df.reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 20-repeated 3-fold split. This creates 24 group members (360 samples) per test set.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()